<a href="https://colab.research.google.com/github/Lucas66666677/SoulSync-AI-Chatting-Bot/blob/main/%F0%9F%94%92_SoulSync_%E7%A7%81%E5%AF%86%E6%97%A5%E8%A8%98%E6%BC%94%E7%A4%BA%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 🔒 SoulSync
# ==========================================

import os
import time
import json
from getpass import getpass

# 💡 安全讀取金鑰的方式：
# 方式 A：手動輸入 (不會儲存在檔案中)
# 方式 B：使用 Colab 的 Secrets (建議)
try:
    from google.colab import userdata
    MY_NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    MY_GROQ_KEY = userdata.get('GROQ_KEY')
except:
    # 如果不在 Colab 或沒設定 Secrets，則執行時詢問
    print("請輸入您的金鑰 (輸入時不會顯示字元)：")
    MY_NGROK_TOKEN = getpass("Ngrok Authtoken: ")
    MY_GROQ_KEY = getpass("Groq API Key: ")

# 1. 安裝環境
print("⏳ 正在安裝環境...")
os.system("pip install -q streamlit pyngrok groq")

from pyngrok import ngrok

# 2. 設定環境變數 (供後續 Streamlit 使用)
os.environ["GROQ_API_KEY"] = MY_GROQ_KEY

# 3. 寫入主程式
with open("app.py", "w", encoding="utf-8") as f:
    f.write(f'''
import streamlit as st
import os
import json
import time
from groq import Groq

# 💡 從環境變數讀取金鑰，不要寫死在字串裡
GROQ_API_KEY = "{MY_GROQ_KEY}"

# --- 1. 系統設定與檔案處理 ---
DATA_FILE = "shared_diary_data.json"

if not os.path.exists(DATA_FILE):
    with open(DATA_FILE, "w") as f: json.dump({{}}, f)

def load_data():
    try:
        with open(DATA_FILE, "r") as f: return json.load(f)
    except: return {{}}

def save_data(data):
    with open(DATA_FILE, "w") as f: json.dump(data, f)

# --- 2. AI 核心邏輯 ---
def get_ai_response(messages):
    # 確保 API Key 被正確帶入
    client = Groq(api_key=GROQ_API_KEY)
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0.6,
            max_tokens=1024,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"AI 連線錯誤: {{str(e)}}"

def analyze_mood_status(text):
    client = Groq(api_key=GROQ_API_KEY)
    system_prompt = """
    你是一個心理健康監測員。請分析使用者的日記內容。
    回傳 JSON 格式：
    1. "status_color": "green" (正常), "yellow" (輕微焦慮), "red" (危險/求救)
    2. "status_message": 給好友的簡短提示 (不洩露內容)。
    """
    try:
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {{"role": "system", "content": system_prompt}},
                {{"role": "user", "content": f"日記內容: {{text}}"}}
            ],
            response_format={{"type": "json_object"}}
        )
        return json.loads(completion.choices[0].message.content)
    except:
        return {{"status_color": "gray", "status_message": "AI 分析中..."}}

# --- 3. Streamlit 介面設計 ---
st.set_page_config(page_title="SoulSync - 私密日記", page_icon="🔒")
st.markdown("""
<style>
    .stChatMessage {{ border-radius: 15px; }}
    .status-box {{ padding: 15px; border-radius: 10px; margin-bottom: 10px; color: white; font-weight: bold;}}
    .status-green {{ background-color: #28a745; }}
    .status-yellow {{ background-color: #ffc107; color: black; }}
    .status-red {{ background-color: #dc3545; }}
</style>
""", unsafe_allow_html=True)

if "messages" not in st.session_state: st.session_state.messages = []
if "user_status" not in st.session_state: st.session_state.user_status = {{"status_color": "green", "status_message": "剛上線"}}

# --- 登入畫面 ---
if "username" not in st.session_state:
    st.title("🔒 SoulSync 私密日記")
    c1, c2 = st.columns(2)
    room_input = c1.text_input("房間代碼 (Room ID)")
    name_input = c2.text_input("您的暱稱")

    if st.button("連線進入"):
        if room_input and name_input:
            st.session_state.room_id = room_input
            st.session_state.username = name_input
            st.session_state.messages.append({{
                "role": "assistant",
                "content": f"嗨 {{name_input}}，我是你的 AI 陪伴。今天過得怎麼樣？"
            }})
            st.rerun()

# --- 主畫面 ---
else:
    room_id = st.session_state.room_id
    current_user = st.session_state.username
    data = load_data()
    if room_id not in data: data[room_id] = {{}}
    data[room_id][current_user] = {{"last_seen": time.time(), "status": st.session_state.user_status}}
    save_data(data)

    with st.sidebar:
        st.title(f"🏠 房間：{{room_id}}")
        st.write(f"👤 您是：{{current_user}}")
        st.divider()
        st.subheader("📡 好友狀態監測")

        partners = [u for u in data[room_id] if u != current_user]
        if not partners:
            st.warning("等待好友連線...")
        else:
            for partner in partners:
                p_data = data[room_id][partner]
                is_online = (time.time() - p_data["last_seen"]) < 60
                online_str = "🟢 在線" if is_online else "⚪ 離線"
                s_color = p_data["status"].get("status_color", "green")
                s_msg = p_data["status"].get("status_message", "尚無資料")

                st.markdown(f"**{{partner}}** ({{online_str}})")
                st.markdown(f'<div class="status-box status-{{s_color}}">{{s_msg}}</div>', unsafe_allow_html=True)

        st.divider()
        if st.button("🔥 銷毀紀錄並登出", type="primary"):
            if room_id in data: del data[room_id]
            save_data(data)
            st.session_state.clear()
            st.rerun()

    st.subheader("📒 AI 引導式日記")
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]): st.write(msg["content"])

    if prompt := st.chat_input("寫下今天的感受..."):
        st.session_state.messages.append({{"role": "user", "content": prompt}})
        with st.chat_message("user"): st.write(prompt)

        with st.status("AI 正在聆聽並分析...", expanded=False):
            analysis = analyze_mood_status(prompt)
            st.session_state.user_status = analysis
            data = load_data()
            if room_id in data:
                data[room_id][current_user]["status"] = analysis
                save_data(data)

        msgs = [{{"role": "system", "content": "你是溫暖的心理日記 AI，請用繁體中文簡短回應，多展現同理心。"}}]
        for m in st.session_state.messages[-5:]: msgs.append({{"role": m["role"], "content": m["content"]}})
        ai_reply = get_ai_response(msgs)

        st.session_state.messages.append({{"role": "assistant", "content": ai_reply}})
        with st.chat_message("assistant"): st.write(ai_reply)
        st.rerun()
    ''')

# 4. 啟動 Streamlit (增加強制 Port 設定與輸出重導向)
print("🚀 正在啟動 Streamlit...")
# 使用 nohup 並將輸出導入日誌，確保在背景穩定執行
os.system("nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &")

# ⏳ 關鍵：等待 Streamlit 暖機 (5秒)
time.sleep(5)

# 5. 啟動 Ngrok
try:
    print("🌐 正在建立 Ngrok 隧道...")
    ngrok.kill() # 先清理舊的連線
    ngrok.set_auth_token(MY_NGROK_TOKEN)

    # 建立連接
    public_url = ngrok.connect(8501, "http").public_url

    print("\n" + "="*50)
    print(f"🎉 成功了！請點擊網址：{public_url}")
    print("💡 如果還是看到 undefined，請等 10 秒後重新整理網頁")
    print("="*50 + "\n")

except Exception as e:
    print(f"❌ Ngrok 啟動失敗: {e}")

⏳ 正在安裝環境...
🚀 正在啟動 Streamlit...
🌐 正在建立 Ngrok 隧道...

🎉 成功了！請點擊網址：https://edyth-lacelike-interestedly.ngrok-free.dev
💡 如果還是看到 undefined，請等 10 秒後重新整理網頁

